# Module 1 - Solving Ax = b: Linear Systems, Least Squares & Conditioning

One question runs under almost every method in this course: given a linear relationship between
a measurement and the quantity that produced it, **solve for the quantity**. That is
$A\mathbf{x} = \mathbf{b}$ -- an *inverse problem*. Biology hands us the measurement $\mathbf{b}$ and a
model $A$ of how an unknown $\mathbf{x}$ produced it, and we reason backward to $\mathbf{x}$. Later modules
are specializations of this one move: nonlinear fits (Module 2), prediction (Module 3), recovery from
too few samples (Module 4), and dynamics (Module 7) are all least-squares solves underneath.

We build that primitive here on a concrete biomedical task: **aligning two medical images**. When
a clinic photographs the same retina on two visits, the images are shifted, rotated, and scaled
relative to each other; before you can compare them you must **register** them -- recover the
geometric transform mapping one onto the other. Given a handful of matched landmark points, that
recovery is *exactly* an $A\mathbf{x} = \mathbf{b}$ least-squares solve.

Three ideas, in order: a **determined** solve (just enough equations to pin the answer down), the
**least-squares** solve (what to do with more, noisy equations than unknowns), and **conditioning**
(reading the **singular values** of $A$ to know when the answer can be trusted).

**Reading.** Kutz, *Data-Driven Modeling & Scientific Computation*, 2nd ed. -- Chapter 2
(linear systems and the direct solution of $A\mathbf{x} = \mathbf{b}$) and Chapter 4, section 1
(least-squares fitting, the normal equations, and the pseudoinverse). Everything below is explained in
our own terms and run against a known-truth fixture.

**Learning goals.**

- Set up and solve a **determined** linear system $A\mathbf{x} = \mathbf{b}$, and say when its solution is unique.
- Solve an **over-determined** system by **least squares** (pseudoinverse / normal equations) and read its residual.
- Recover a *known* image transform from noisy landmark correspondences and score the recovery against the truth you set.
- Use the **condition number** -- the singular-value spread of $A$ -- to judge when a solve is trustworthy, and watch it fail on collinear landmarks.
- Close with an explicit claim-and-limitations statement.

```{admonition} Which paradigm?
:class: note
**Model-driven form, data-driven parameters.** Registration commits to a *model form* up front --
an **affine** map (rotation, scale, shear, translation), six numbers -- because repositioning a
camera or a slide is, to first order, affine. That is the deductive choice. The six numbers are
then recovered *from the data* (the matched landmarks) by least squares -- the inductive step. You
meet this pairing all term: choose a structured form when you know it, and let the data fill in
its parameters.
```

In [ ]:
import numpy as np

import ddm4bio
from ddm4bio import seed_everything
from ddm4bio.viz.style import set_style

seed_everything()
set_style()

print(f"ddm4bio version: {ddm4bio.__version__}")

## 1. A determined solve: just enough equations

A 2-D **affine** transform sends a point $(x, y)$ to

$$
x' = a\,x + b\,y + c, \qquad y' = d\,x + e\,y + f,
$$

so it has **six** unknowns. Each landmark correspondence $(x, y) \to (x', y')$ gives **two**
equations, so **three** non-collinear landmarks make exactly six equations in six unknowns -- a
**determined** system $A\mathbf{t} = \mathbf{b}$ with a unique solution that `numpy.linalg.solve`
returns directly. We stack the two rows per landmark into the design matrix $A$.

In [ ]:
def affine_design(pts):
    """Two rows per landmark, stacking x' = a*x + b*y + c and y' = d*x + e*y + f."""
    n = len(pts)
    A = np.zeros((2 * n, 6))
    A[0::2, 0], A[0::2, 1], A[0::2, 2] = pts[:, 0], pts[:, 1], 1.0
    A[1::2, 3], A[1::2, 4], A[1::2, 5] = pts[:, 0], pts[:, 1], 1.0
    return A


def apply_affine(t, pts):
    """Map points by the affine parameter vector t = [a, b, c, d, e, f]."""
    return (t.reshape(2, 3) @ np.c_[pts, np.ones(len(pts))].T).T


# The known-truth transform we will recover (a small rotation, scale, and translation).
T_true = np.array([[1.05, -0.18, 35.0],
                   [0.18, 0.95, -20.0]])
t_true = T_true.reshape(-1)

# Three non-collinear landmarks -> exactly six equations in six unknowns: a determined solve.
three = np.array([[120.0, 140.0], [900.0, 250.0], [400.0, 1000.0]])
A3 = affine_design(three)
t_determined = np.linalg.solve(A3, apply_affine(t_true, three).reshape(-1))

print(f"design matrix A: {A3.shape[0]} equations x {A3.shape[1]} unknowns")
print(f"unique solution recovers the truth exactly (max |error| = {np.abs(t_determined - t_true).max():.1e})")

## 2. More equations than unknowns: least squares

Real data gives you *more* than three landmarks, each measured with error, so the $2n \times 6$
system is **over-determined** -- no $\mathbf{t}$ satisfies every equation. **Least squares** returns
the closest fit, the $\mathbf{t}$ minimizing $\lVert A\mathbf{t} - \mathbf{b}\rVert^2$. Geometrically it
**projects** $\mathbf{b}$ onto the column space of $A$; algebraically that minimizer is
$\mathbf{t} = A^{+}\mathbf{b}$ with the **pseudoinverse** $A^{+}$, which solves the normal equations
$A^\top A\,\mathbf{t} = A^\top\mathbf{b}$. `numpy.linalg.lstsq` computes it stably via the SVD, without
ever forming $A^\top A$.

In [ ]:
rng = np.random.default_rng(0)
n_landmarks = 15
noise_px = 2.0
marks = rng.uniform([100, 100], [1300, 1300], size=(n_landmarks, 2))
targets = apply_affine(t_true, marks) + rng.normal(0.0, noise_px, marks.shape)

A = affine_design(marks)
t_ls, _, rank, svals = np.linalg.lstsq(A, targets.reshape(-1), rcond=None)
recovered_rmse = float(np.sqrt(np.mean((apply_affine(t_ls, marks) - targets) ** 2)))

print(f"over-determined system: {A.shape[0]} equations, {A.shape[1]} unknowns (rank {rank})")
print(f"least-squares landmark residual : {recovered_rmse:.2f} px  (measurement noise was {noise_px:.1f} px)")
print(f"parameter recovery vs known truth: max |error| = {np.abs(t_ls - t_true).max():.3f}")

## 3. Registering a real retinal image

Now the task itself. We take a **real retinal fundus photograph**, apply the known transform to
make a "second-visit" image, and recover the transform from the landmark correspondences by least
squares. Warping the moving image by the *recovered* transform should bring it back into register
with the original -- and since we set the true transform, we can score the recovery exactly.

In [ ]:
import matplotlib.pyplot as plt
from skimage import data
from skimage.color import rgb2gray
from skimage.transform import AffineTransform, warp

fixed = rgb2gray(data.retina())                    # a real retinal fundus photograph
tf_true = AffineTransform(matrix=np.vstack([T_true, [0, 0, 1]]))
moving = warp(fixed, tf_true.inverse, order=1, preserve_range=True)   # the simulated second visit

tf_rec = AffineTransform(matrix=np.vstack([t_ls.reshape(2, 3), [0, 0, 1]]))
registered = warp(moving, tf_rec, order=1, preserve_range=True)       # undo it with the LS fit

valid = (fixed > 0.05) & (registered > 0.05)
mse_before = float(np.mean((fixed - moving)[fixed > 0.05] ** 2))
mse_after = float(np.mean((fixed - registered)[valid] ** 2))
print(f"retinal image {fixed.shape}; misalignment MSE {mse_before:.4f} -> {mse_after:.4f} "
      f"after registration ({mse_before / mse_after:.0f}x lower)")

fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), constrained_layout=True)
for ax, im, title in zip(
    axes, [fixed, moving, registered],
    ["fixed (visit 1)", "moving (visit 2, misaligned)", "registered (least-squares fit)"],
):
    ax.imshow(im, cmap="gray")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
axes[0].scatter(marks[:, 0], marks[:, 1], s=16, c="#39ff14", edgecolor="black", linewidth=0.4)
fig;

The registered image snaps back onto the fixed one -- the green points in the first panel are the
landmark correspondences we solved from, and the misalignment error drops by two orders of
magnitude. Because we *chose* the true transform, that recovery is measured against a real answer,
not eyeballed.

## 4. When can a solve be trusted? Conditioning

Least squares always returns *an* answer, but not always a *reliable* one. The solve effectively
divides by the **singular values** of $A$; when the smallest is tiny, a small error in the
landmarks is amplified into a large error in the transform. The **condition number**
$\kappa(A) = \sigma_{\max} / \sigma_{\min}$ measures that amplification.

Two different things inflate $\kappa$ here, and separating them is the point of this section.
The first is **scaling**: our design matrix sets pixel coordinates of order $10^3$ beside a
column of ones, and that mismatch alone pushes $\kappa$ into the thousands even for ideal
landmarks. That is a statement about the units we chose, not about the problem, and rescaling
the coordinates removes it. The second is **geometry**: landmarks strung along one line -- a
single vessel, say -- make the columns of $A$ nearly dependent, $\sigma_{\min} \to 0$, and
$\kappa$ explodes for a reason no change of units can fix. Only the second is a property of the
problem, and it is the one that destroys the recovered transform. Below we change only the
landmark geometry -- same math, same noise -- and then re-measure both systems in normalized
coordinates to see which part of the damage survives.

In [ ]:
def cond_and_error(pts, seed):
    r = np.random.default_rng(seed)
    tgt = apply_affine(t_true, pts) + r.normal(0.0, noise_px, pts.shape)
    Ad = affine_design(pts)
    t_hat, *_ = np.linalg.lstsq(Ad, tgt.reshape(-1), rcond=None)
    return float(np.linalg.cond(Ad)), float(np.abs(t_hat - t_true).max())


collinear = np.c_[np.linspace(150, 1250, n_landmarks), 700 + rng.normal(0.0, 3.0, n_landmarks)]
kappa_ok, err_ok = cond_and_error(marks, seed=1)
kappa_bad, err_bad = cond_and_error(collinear, seed=1)
print(f"well-spread landmarks   : condition number {kappa_ok:8.0f} -> max parameter error {err_ok:.2f}")
print(f"near-collinear landmarks: condition number {kappa_bad:8.0f} -> max parameter error {err_bad:.2f}")


# Separate the two sources of ill-conditioning: rescale the landmarks to zero mean and
# mean distance sqrt(2) (the standard normalization for this solve) and re-measure.
# What survives the rescaling is geometry; what vanishes was only our choice of units.
def normalized(pts):
    centered = pts - pts.mean(axis=0)
    return centered * (np.sqrt(2.0) / np.linalg.norm(centered, axis=1).mean())


kappa_ok_n = float(np.linalg.cond(affine_design(normalized(marks))))
kappa_bad_n = float(np.linalg.cond(affine_design(normalized(collinear))))
print(f"  rescaled to normalized coordinates: {kappa_ok_n:6.1f} (well-spread) "
      f"vs {kappa_bad_n:8.1f} (near-collinear)")
print(f"  -> units alone accounted for a factor of ~{kappa_ok / kappa_ok_n:,.0f} on the well-spread\n"
      f"     system; the ~{kappa_bad_n / kappa_ok_n:,.0f}x gap that remains is geometry, and no\n"
      f"     rescaling removes it.")

fig, axes = plt.subplots(1, 2, figsize=(9, 4.4), constrained_layout=True)
for ax, pts, name, kappa in [(axes[0], marks, "well-spread", kappa_ok),
                             (axes[1], collinear, "near-collinear", kappa_bad)]:
    ax.scatter(pts[:, 0], pts[:, 1], s=30, c="#1f77b4", edgecolor="black", linewidth=0.4)
    ax.set_xlim(0, 1411)
    ax.set_ylim(1411, 0)
    ax.set_aspect("equal")
    ax.set_title(f"{name} landmarks\n$\\kappa(A)$ = {kappa:,.0f}", fontsize=10)
    ax.set_xlabel("x (px)")
    ax.set_ylabel("y (px)")
fig;

## 5. Interpretation

Every ddm4bio analysis closes with an explicit claim and its named limitations. Here the claim is
about the recovered transform and about when the solve can be trusted.

In [ ]:
from ddm4bio.interpret import interpretation_block, show_interpretation

block = interpretation_block(
    claim=(
        f"From {n_landmarks} well-spread landmarks, least squares recovers the known affine "
        f"transform to within {np.abs(t_ls - t_true).max():.2f} on every parameter, with a "
        f"{recovered_rmse:.1f}-pixel residual at the {noise_px:.0f}-pixel noise floor, and it "
        f"re-aligns the real retinal image ({mse_before / mse_after:.0f}x lower misalignment). "
        f"The identical solve on near-collinear landmarks is not trustworthy: the condition number "
        f"rises from {kappa_ok:.0f} to {kappa_bad:.0f} and the parameter error from {err_ok:.1f} to "
        f"{err_bad:.0f}."
    ),
    limitations_list=[
        "The transform is affine by construction, so this scores the solver, not whether a real "
        "visit-to-visit change is affine -- tissue deformation and lens distortion add non-affine "
        "warps six parameters cannot capture.",
        "Landmark correspondences are given here; detecting and matching them on real images is a "
        "separate, harder problem.",
        "The noise is i.i.d. Gaussian on landmark coordinates; real localization error is "
        "heavier-tailed and spatially correlated.",
    ],
)
show_interpretation(block)

## Exercises

- **Rigid, not affine.** An affine fit has six free parameters; a *rigid* transform (rotation +
  translation only, no scale or shear) has three. Constrain the fit to rigid and compare its
  residual and its stability to the affine fit -- when is affine's extra flexibility worth it, and
  when is it just fitting noise?
- **Weight the landmarks.** Some landmarks are localized more precisely than others. Re-solve the
  over-determined system as *weighted* least squares, down-weighting noisier points by
  $1/\sigma^2$, and check whether the recovered transform improves.
- **Engineer an ill-conditioned solve.** Move the landmarks until $\kappa(A)$ crosses $10^4$, then
  perturb one landmark by a single pixel and watch the recovered transform swing. For a fixed
  number of points, what landmark geometry gives the *best*-conditioned solve?